In [ ]:
# Import packages
from packages import *

plt.style.use("~/geoscience/albedo_downscaling/MNRAS.mplstyle")
%matplotlib inline

In [ ]:
cf_file = "/bsuhome/tnde/scratch/felix/Sentinel-2/s2_albedo_outputs/tsi_cloud_fractions.csv"
cf_vals = pd.read_csv(cf_file)
print(len(cf_vals))
cf_vals = cf_vals[cf_vals["cf_interp"]<=0.40]
cf_vals = cf_vals.drop_duplicates(subset=["date"])
display(cf_vals.head())
len(cf_vals)

In [ ]:
albedo_path = "/bsuhome/tnde/scratch/felix/Sentinel-2/s2_albedo_outputs/*_S2_BLUE20m_SW_hard.tif"
albedo_files = os.path.abspath(albedo_path)
albedo_files_sorted = sorted(glob.glob(albedo_files))
unet_files_list = []
# invalid_dates = ["2021-09-23", "2022-03-12", "2022-03-12", 
#                  "2022-04-06", "2022-05-06", "2022-05-26", 
#                  "2022-09-18", "2022-10-13", "2022-10-18"]

# invalid_dates = ["2022-07-10", "2022-05-06", "2022-04-21", 
#                  "2022-04-06", "2022-04-01", "2022-03-12", 
#                  "2021-09-23"]

invalid_dates = ["2022-09-23", "2021-10-28", "2022-03-12", "2022-04-01", 
                 "2022-04-06", "2022-04-21", "2022-04-26", "2022-05-06", 
                 "2022-05-26", "2022-05-31", "2022-07-10"]

for s2_date in list(cf_vals["date"]):
    unet_file = f"/bsuhome/tnde/scratch/felix/Sentinel-2/s2_albedo_outputs/{s2_date}_S2_BLUE20m_SW_hard.tif"
    if s2_date in invalid_dates:
        pass
    elif unet_file not in albedo_files_sorted:
        print(f"File: {unet_file} not found in list.")
    else:
        unet_files_list.append(unet_file)
print(f"Number of U-Net files: {len(unet_files_list)}")

In [ ]:
# before dropping duplicates
data = {
    "Percentage": [5, 10, 25, 50, 80, 100],
    "Sentinel-2": [41, 57, 81, 118, 162, 185],
    "TSI":        [5, 50, 68, 102, 123, 185],
}

# # after dropping duplicates
# data = {
#     "Percentage": [5, 10, 25, 50, 80, 100],
#     "Sentinel-2": [29, 39, 56, 81, 111, 128],
#     "TSI":        [4, 36, 48, 71, 85, 128],
# }


df = pd.DataFrame(data)

# # Save to CSV 
# csv_path = "/bsuhome/tnde/scratch/felix/Sentinel-2/s2_albedo_outputs/ccf_vs_no_of_images_without_duplicates.csv"
# df.to_csv(csv_path, index=False)

# Plot
plt.figure(figsize=(8, 5))
plt.plot(df["Percentage"], df["Sentinel-2"], marker="o", label="Sentinel-2")
plt.plot(df["Percentage"], df["TSI"], marker="s", label="TSI")
plt.xlabel("Cloud cover percentage (%)")
plt.ylabel("Number of valid images")
# plt.title("Cloud cover percentage vs number of valid images")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
png_path = "/bsuhome/tnde/scratch/felix/Sentinel-2/s2_albedo_outputs/ccf_vs_no_of_images.png"
plt.savefig(png_path, dpi=150, bbox_inches="tight")
plt.show()


## Rename GOES NetCDF files.

Uncomment the cell below and run it when necessary. 

Set `DRY_RUN` to `False` only when necessary.

In [ ]:
# from pathlib import Path
# import re
# from datetime import datetime

# DIR = Path("/bsuhome/tnde/scratch/felix/GOES/data/nan_data_new/")
# DRY_RUN = True  # set False to actually rename

# # Looks for "..._eYYYYDDDHHMMSSS..." anywhere in the filename
# END_RE = re.compile(r"_e(\d{4})(\d{3})(\d{2})(\d{2})(\d{2})(\d)(?=_|\.|$)")

# for f in sorted(DIR.glob("*.tif")):
#     m = END_RE.search(f.name)
#     if not m:
#         print(f"SKIP (no end timestamp found): {f.name}")
#         continue

#     # ---- EDIT START: hour filter (~18 UTC only) ----
#     hour = int(m.group(3))  # HH from eYYYYDDDHH...
#     if hour != 18:
#         print(f"PASS (end hour {hour} != 18): {f.name}")
#         continue
#     # ---- EDIT END ----

#     year, doy = m.group(1), m.group(2)
#     date_str = datetime.strptime(year + doy, "%Y%j").strftime("%Y-%m-%d")

#     # Keep the stable prefix up to "_s" (e.g., "OR_ABI-L2-LSAC-M6_G16")
#     base = f.name.split("_s", 1)[0]

#     new_name = f"{base}_{date_str}{f.suffix}"  # .tif
#     target = f.with_name(new_name)

#     # Avoid overwriting if multiple files map to same date
#     if target.exists() and target.resolve() != f.resolve():
#         stem = target.stem
#         k = 2
#         while True:
#             candidate = target.with_name(f"{stem}_{k}{f.suffix}")
#             if not candidate.exists():
#                 target = candidate
#                 break
#             k += 1

#     if DRY_RUN:
#         print(f"{f.name}  ->  {target.name}")
#     else:
#         f.rename(target)
#         print(f"RENAMED: {f.name}  ->  {target.name}")

In [ ]:
# # --- edit this ---
# DIR = Path("/bsuhome/tnde/scratch/felix/GOES/data/500m-raster")
# DRY_RUN = True   # set to False to actually rename
# # --------------

# pat = re.compile(r"^(\d{2})-(\d{2})-(\d{4})-GOES-500m\.tif$", re.IGNORECASE)

# for f in sorted(DIR.glob("*.tif")):
#     m = pat.match(f.name)
#     if not m:
#         print(f"SKIP (name doesn't match): {f.name}")
#         continue

#     mm, dd, yyyy = m.group(1), m.group(2), m.group(3)
#     new_name = f"{yyyy}-{mm}-{dd}-GOES-500m.tif"
#     target = f.with_name(new_name)

#     # avoid overwriting if the target already exists
#     if target.exists() and target.resolve() != f.resolve():
#         stem = target.stem
#         k = 2
#         while True:
#             candidate = target.with_name(f"{stem}_{k}{target.suffix}")
#             if not candidate.exists():
#                 target = candidate
#                 break
#             k += 1

#     if DRY_RUN:
#         print(f"{f.name}  ->  {target.name}")
#     else:
#         f.rename(target)
#         print(f"RENAMED: {f.name}  ->  {target.name}")

## Matching Sentinel-2 and MODIS files.

### Matching train files

In [ ]:
datetime(2022, 1, 5)

In [ ]:
goes_train_path = "/bsuhome/tnde/scratch/felix/GOES/data/500m-raster/*.tif"
goes_train_files = os.path.abspath(goes_train_path)
goes_train_files_sorted = sorted(glob.glob(goes_train_files))
goes_unet_train_files_list = []

s2_train_path = "/bsuhome/tnde/scratch/felix/Sentinel-2/s2_albedo_outputs/*_S2_BLUE20m_SW_hard.tif"
s2_train_files = os.path.abspath(s2_train_path)
s2_train_files_sorted = sorted(glob.glob(s2_train_files))
s2_unet_train_files_list = []
# invalid_train_dates = ["2021-09-23", "2022-03-12", "2022-03-12", 
#                        "2022-04-06", "2022-05-06", "2022-05-26", 
#                        "2022-09-18", "2022-10-13", "2022-10-18"]

# invalid_train_dates = ["2022-07-10", "2022-05-06", "2022-04-21", 
#                        "2022-04-06", "2022-04-01", "2022-03-12", 
#                        "2021-09-23"]

invalid_train_dates = ["2022-09-23", "2021-10-28", "2022-03-12", "2022-04-01", 
                       "2022-04-06", "2022-04-21", "2022-04-26", "2022-05-06", 
                       "2022-05-26", "2022-05-31", "2022-07-10"]

for s2_date in list(cf_vals["date"]):
    goes_unet_train_file = f"/bsuhome/tnde/scratch/felix/GOES/data/500m-raster/{s2_date}-GOES-500m.tif"
    s2_unet_train_file = f"/bsuhome/tnde/scratch/felix/Sentinel-2/s2_albedo_outputs/{s2_date}_S2_BLUE20m_SW_hard.tif"
    if s2_date in invalid_train_dates:
        pass
    elif goes_unet_train_file not in goes_train_files_sorted:
        # print(f"File: {modis_unet_train_file} not matched.")
        pass
    else:
        goes_unet_train_files_list.append(goes_unet_train_file)
        s2_unet_train_files_list.append(s2_unet_train_file)
goes_unet_train_files_list = goes_unet_train_files_list[:-12]
s2_unet_train_files_list = s2_unet_train_files_list[:-12]
print(f"Number of GOES U-Net train files: {len(goes_unet_train_files_list)}")
print(f"Number of Sentinel-2 U-Net train files: {len(s2_unet_train_files_list)}")

In [ ]:
all_goes_train = goes_unet_train_files_list
all_s2_train    = s2_unet_train_files_list

n_total   = len(all_goes_train)
test_size = max(1, int(0.2 * n_total))   # avoid 0 for tiny datasets
goes_test = all_goes_train[-test_size:]
s2_test    = all_s2_train[-test_size:]
goes_train = all_goes_train[:-test_size]
s2_train    = all_s2_train[:-test_size]
len(s2_train), len(s2_test), len(goes_train), len(goes_test)
# int(0.85*17)

### Matching test files

In [ ]:
goes_test_path = "/bsuhome/tnde/scratch/felix/GOES/data/500m-raster/*.tif"
goes_test_files = os.path.abspath(goes_test_path)
goes_test_files_sorted = sorted(glob.glob(goes_test_files))
goes_unet_test_files_list = []
# invalid_test_dates = ["2021-09-23", "2022-03-12", "2022-03-12", 
#                       "2022-04-06", "2022-05-06", "2022-05-26", 
#                       "2022-09-18", "2022-10-13", "2022-10-18"]

# invalid_test_dates = ["2022-07-10", "2022-05-06", "2022-04-21", 
#                       "2022-04-06", "2022-04-01", "2022-03-12", 
#                       "2021-09-23"]

invalid_test_dates = ["2022-09-23", "2021-10-28", "2022-03-12", "2022-04-01", 
                      "2022-04-06", "2022-04-21", "2022-04-26", "2022-05-06", 
                      "2022-05-26", "2022-05-31", "2022-07-10"]

s2_test_path = "/bsuhome/tnde/scratch/felix/Sentinel-2/s2_albedo_outputs/*_S2_BLUE20m_SW_hard.tif"
s2_test_files = os.path.abspath(s2_test_path)
s2_test_files_sorted = sorted(glob.glob(s2_test_files))
s2_unet_test_files_list = []

for s2_date in list(cf_vals["date"]):
    goes_unet_test_file = f"/bsuhome/tnde/scratch/felix/GOES/data/500m-raster/{s2_date}-GOES-500m.tif"
    s2_unet_test_file = f"/bsuhome/tnde/scratch/felix/Sentinel-2/s2_albedo_outputs/{s2_date}_S2_BLUE20m_SW_hard.tif"
    if s2_date in invalid_test_dates:
        pass
    elif goes_unet_test_file not in goes_test_files_sorted:
        # print(f"File: {modis_unet_test_file} not matched.")
        pass
    else:
        goes_unet_test_files_list.append(goes_unet_test_file)
        s2_unet_test_files_list.append(s2_unet_test_file)
goes_unet_test_files_list = goes_unet_test_files_list[-12:]
s2_unet_test_files_list = s2_unet_test_files_list[-12:]
print(f"Number of GOES U-Net test files: {len(goes_unet_test_files_list)}")
print(f"Number of Sentinel-2 U-Net test files: {len(s2_unet_test_files_list)}")

In [ ]:
goes_unet_test_files_list

In [ ]:
s2_unet_test_files_list

## Rub U-Net

In [ ]:
train_model = True
if train_model:
    history = %run goes_s2_unet.py
else:
    # Clip path to all helper functions
    function_path = os.path.expanduser("~/geoscience/albedo_downscaling/goes_s2_downscaling")
    sys.path.append(function_path)
    # import all the helper functions.
    from goes_s2_unet import *

In [ ]:
TF_HISTORY_PATH = "/bsuhome/tnde/scratch/felix/unet_goes_s2/Results/training/history.json"

In [ ]:
with open(TF_HISTORY_PATH, 'r') as file:
    history_dict = json.load(file)
    
training_loss = history_dict['loss']
validation_loss = history_dict['val_loss']

plt.figure(figsize=(10, 5))
plt.plot(training_loss, label='Training loss')
plt.plot(validation_loss, label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
tif = "/bsuhome/tnde/scratch/felix/unet_goes_s2/Unet_test_preds_s2_new/predicted_s2_2022-12-17_S2_BLUE20m_SW_hard.tif"

import rasterio as rio

with rio.open(tif) as ds:
    arr = ds.read(1, masked=True)  # nodata -> mask
    extent = plotting_extent(ds)

plt.figure(figsize=(8,6))
im = plt.imshow(arr, vmin=0, vmax=1, extent=extent, origin="upper")
plt.tick_params(axis='x', rotation=45)
plt.title("Sentinel-2 Shortwave Albedo (keep-all)")
plt.xlabel(ds.crs.to_string()); plt.ylabel("y (map units)")
cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.set_label("Albedo (0–1)")
plt.tight_layout(); plt.show()

In [ ]:
# tif1 = "/bsuhome/tnde/scratch/felix/UNet2/Unet_test_preds_s2_new/predicted_s2_2022-02-20_S2_BLUE20m_SW_hard.tif"
# tif2 = "/bsuhome/tnde/scratch/felix/Sentinel-2/s2_albedo_outputs/2022-02-20_S2_BLUE20m_SW_hard.tif"#s2_unet_test_files_list[0]

# fig, axs = plt.subplots(1, 2, figsize=(12,5), constrained_layout=True)
# for ax, path in zip(axs, [tif1, tif2]):
#     with rio.open(path) as ds:
#         arr = ds.read(1, masked=True)
#         extent = plotting_extent(ds)
#     im = ax.imshow(arr, vmin=0, vmax=1, extent=extent, origin="upper")
#     ax.tick_params(axis='x', rotation=0)
#     # ax.set_title(f"{date} • SW albedo • {title}")
#     ax.set_xlabel(ds.crs.to_string()); ax.set_ylabel("")

# fig.suptitle("Sentinel-2 Shortwave Albedo (keep-all)")
# fig.colorbar(im, ax=axs, location="right", fraction=0.046, pad=0.04, label="Albedo (0–1)")
# plt.show()

In [ ]:
# --- Directories with TIFs ---
dir_pred = Path("/bsuhome/tnde/scratch/felix/unet_goes_s2/Unet_test_preds_s2_new")
dir_ref  = Path("/bsuhome/tnde/scratch/felix/Sentinel-2/s2_albedo_outputs")

# --- Helper to extract YYYY-MM-DD from filename ---
date_re = re.compile(r"\d{4}-\d{2}-\d{2}")

def extract_date(path: Path):
    m = date_re.search(path.name)
    return m.group(0) if m else None

# --- Index files in each directory by date ---
pred_index = {}
for p in sorted(dir_pred.glob("*.tif")):
    d = extract_date(p)
    if d is not None:
        pred_index.setdefault(d, []).append(p)

ref_index = {}
for p in sorted(dir_ref.glob("*.tif")):
    d = extract_date(p)
    if d is not None:
        ref_index.setdefault(d, []).append(p)

# --- Find common dates ---
common_dates = sorted(set(pred_index.keys()) & set(ref_index.keys()))
print(f"Found {len(common_dates)} matching dates")

# --- Loop over matching dates and plot pairs ---
for d in common_dates:
    # if multiple files per date, just take the first of each
    pred_path = pred_index[d][0]
    ref_path  = ref_index[d][0]

    fig, axs = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

    for ax, path, title in zip(
        axs,
        [ref_path, pred_path],
        ["Reference S2 SW albedo", "Predicted S2 SW albedo"],
    ):
        with rio.open(path) as ds:
            arr = ds.read(1, masked=True)
            extent = plotting_extent(ds)

        im = ax.imshow(
            arr,
            extent=extent,
            origin="upper"
        )
        ax.set_title(f"{title}\n{d}")
        # ax.set_xlabel(ds.crs.to_string())
        ax.set_xlabel("Easting (m)")
        ax.set_ylabel("Northing (m)")

    # fig.suptitle(f"Sentinel-2 Shortwave Albedo (keep-all) — {d}")
    fig.colorbar(im, ax=axs, location="right",
                 fraction=0.046, pad=0.04, label="Albedo (0–1)")
    plt.show()


In [ ]:
tif1 = "/bsuhome/tnde/scratch/felix/GOES/data/nan_data_new/OR_ABI-L2-LSAC-M6_G16_2022-04-26.tif"
tif2 = "/bsuhome/tnde/scratch/felix/Sentinel-2/s2_albedo_outputs/2022-04-26_S2_BLUE20m_SW_hard.tif"#s2_unet_test_files_list[0]

fig, axs = plt.subplots(1, 2, figsize=(12,5), constrained_layout=True)
for ax, path in zip(axs, [tif1, tif2]):
    with rio.open(path) as ds:
        arr = ds.read(1, masked=True)
        extent = plotting_extent(ds)
    im = ax.imshow(arr, vmin=0, vmax=1, extent=extent, origin="upper")
    ax.tick_params(axis='x', rotation=0)
    # ax.set_title(f"{date} • SW albedo • {title}")
    crs_label = ds.crs.to_string() if ds.crs is not None else "CRS: None"
    ax.set_xlabel(crs_label); ax.set_ylabel("")

fig.suptitle("Sentinel-2 Shortwave Albedo (keep-all)")
fig.colorbar(im, ax=axs, location="right", fraction=0.046, pad=0.04, label="Albedo (0–1)")
plt.show()

In [ ]:
tif = f"/bsuhome/tnde/scratch/felix/GOES/data/nan_data_new/OR_ABI-L2-LSAC-M6_G16_2022-03-02.tif"
import rasterio as rio

with rio.open(tif) as ds:
    arr = ds.read(1, masked=True)  # nodata -> mask
    extent = plotting_extent(ds)

plt.figure(figsize=(8,6))
im = plt.imshow(arr, vmin=0, vmax=1, extent=extent, origin="upper")
plt.tick_params(axis='x', rotation=45)
# plt.title("Sentinel-2 Shortwave Albedo (keep-all)")
plt.xlabel(ds.crs.to_string()); plt.ylabel("y (map units)")
cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.set_label("Albedo (0–1)")
plt.tight_layout(); plt.show()

In [ ]:
tif1 = f"/bsuhome/tnde/scratch/felix/GOES/data/nan_data_new/OR_ABI-L2-LSAC-M6_G16_2022-11-07.tif"
tif2 = "/bsuhome/tnde/scratch/felix/Sentinel-2/s2_albedo_outputs/2022-11-07_S2_BLUE20m_SW_hard.tif"#s2_unet_test_files_list[0]

fig, axs = plt.subplots(1, 2, figsize=(12,5), constrained_layout=True)
for ax, path in zip(axs, [tif1, tif2]):
    with rio.open(path) as ds:
        arr = ds.read(1, masked=True)
        extent = plotting_extent(ds)
    im = ax.imshow(arr, vmin=0, vmax=1, extent=extent, origin="upper")
    ax.tick_params(axis='x', rotation=0)
    # ax.set_title(f"{date} • SW albedo • {title}")
    ax.set_xlabel(ds.crs.to_string() if ds.crs is not None else ""); ax.set_ylabel("")

fig.suptitle("Sentinel-2 Shortwave Albedo (keep-all)")
fig.colorbar(im, ax=axs, location="right", fraction=0.046, pad=0.04, label="Albedo (0–1)")
plt.show()

In [ ]:
tif = "/bsuhome/tnde/scratch/felix/GOES/data/500m-raster/12-21-2021-GOES-500m.tif"
import rasterio as rio

with rio.open(tif) as ds:
    arr = ds.read(1, masked=True)  # nodata -> mask
    extent = plotting_extent(ds)

plt.figure(figsize=(8,6))
im = plt.imshow(arr, vmin=0, vmax=1, extent=extent, origin="upper")
plt.tick_params(axis='x', rotation=45)
plt.title("Sentinel-2 Shortwave Albedo (keep-all)")
plt.xlabel(ds.crs.to_string()); plt.ylabel("y (map units)")
cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.set_label("Albedo (0–1)")
plt.tight_layout(); plt.show()

In [ ]:
tif = f"/bsuhome/tnde/scratch/felix/GOES/data/nan_data_new/OR_ABI-L2-LSAC-M6_G16_2022-12-17.tif"
import rasterio as rio

with rio.open(tif) as ds:
    arr = ds.read(1, masked=True)  # nodata -> mask
    extent = plotting_extent(ds)

plt.figure(figsize=(8,6))
im = plt.imshow(arr, vmin=0, vmax=1, extent=extent, origin="upper")
plt.tick_params(axis='x', rotation=45)
plt.title("Sentinel-2 Shortwave Albedo (keep-all)")
plt.xlabel(ds.crs.to_string()); plt.ylabel("y (map units)")
cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.set_label("Albedo (0–1)")
plt.tight_layout(); plt.show()